# Capacitive Sensor Calibration Notebook

This notebook records how much a capacitive pressure sensor's capacitance changes when known weights are placed on it, then fits a calibration curve so you can convert future capacitance readings into grams.

It works with **any set of weights** — you choose them in Step 2.

## What you need
- An ESP32 running the sensor firmware with `CSV_MODE = true` (it prints lines like `t_s,ticks,delta_ticks,cap_pF,delta_pF,timeouts` and re-baselines when it receives `b`)
- A USB cable and the ESP32's serial port name (Windows: `COM3`, `COM11`…; Mac: `/dev/cu.usbserial-…`; Linux: `/dev/ttyUSB0`)
- A set of known weights (calibration masses, or objects you weighed on a scale)
- Python with Jupyter (Anaconda works)

## How to use this notebook
Run the cells **top to bottom** with **Shift + Enter**. Each step has a short explanation above it. You only need to edit the settings cell in Step 2.

| Step | What happens |
|---|---|
| 1 | Install the two required libraries (once per computer) |
| 2 | Enter your port, file name, and the weights you'll use |
| 3 | Connect to the ESP32 |
| 4 | Load the helper functions |
| 5 | Zero the sensor with nothing on it |
| 6 | Record each weight (guided, or one at a time) |
| 7 | See the results and the calibration curve |
| 8 | Convert new readings to grams |
| 9 | Disconnect |

> **Tip:** Close the Arduino Serial Monitor before Step 3 — only one program can use the port at a time.

## Step 1 — Install libraries
Run this once per computer. If it says *"you may need to restart the kernel"*, click **Kernel → Restart** and continue from Step 2.

In [ ]:
%pip install pyserial matplotlib numpy

## Step 2 — Your settings ✏️ (the only cell you need to edit)

- **`PORT`** — the ESP32's serial port. Not sure? Run the cell below this one to list available ports.
- **`CSV_FILE`** — where readings are saved. Use a **new name for each test run** (e.g. `run1`, `run2`) so data from different setups doesn't mix. Re-running with the same name *adds* to the file.
- **`WEIGHTS_G`** — the weights you will place on the sensor, in grams, in the order you'll place them. Include `0` first so you capture the empty reading. Use as many or as few as you like.
- **`REPEATS`** — how many times to go through the whole list. 2–3 repeats give you a sense of repeatability (hysteresis, drift).
- **`SECONDS`** — how long to record each weight. **`SETTLE`** — how long to wait after placing the weight before recording.

In [ ]:
PORT      = "COM11"                       # your ESP32's serial port
BAUD      = 115200                        # must match Serial.begin() in the firmware
CSV_FILE  = "calibration_run1.csv"        # new name for each test run

WEIGHTS_G = [0, 100, 200, 300, 400, 500, 600, 700, 800, 900]   # grams, in the order you'll place them
REPEATS   = 1                             # times to go through the list

SECONDS   = 10                            # seconds of data per weight
SETTLE    = 2                             # seconds to wait after placing a weight

In [ ]:
# Optional: list the serial ports on this computer (helps find PORT)
from serial.tools import list_ports
for p in list_ports.comports():
    print(f"{p.device:15s} {p.description}")

## Step 3 — Connect to the ESP32
**Make sure nothing is on the sensor**, then run this cell. The ESP32 resets when the port opens and takes a fresh baseline, so we wait 3 seconds.

If you get *"could not open port"* / *"Access is denied"*: check the port name, and close the Arduino Serial Monitor or any other notebook using it.

In [ ]:
import csv, os, time
from datetime import datetime
import serial
import numpy as np
import matplotlib.pyplot as plt

COLS = ["timestamp", "run", "weight_g", "t_s", "ticks", "delta_ticks", "cap_pF", "delta_pF", "timeouts"]

try:
    ser.close()        # if this cell is re-run, close the old connection first
except NameError:
    pass
ser = serial.Serial(PORT, BAUD, timeout=0.1)
print(f"Connected to {PORT}. Waiting for the ESP32 baseline - don't touch the sensor...")
time.sleep(3)
ser.reset_input_buffer()
print("Ready.")

## Step 4 — Load the helper functions
Just run this cell. It defines:
- `rezero()` — takes a new baseline (sensor must be empty)
- `record(weight_g)` — records readings for a weight that is already on the sensor and saves them to `CSV_FILE`
- `calibrate()` — walks you through every weight in `WEIGHTS_G`, prompting you each time

In [ ]:
def _parse(line):
    parts = line.strip().split(",")
    if len(parts) != 6 or parts[1] == "TIMEOUT":
        return None
    try:
        return [float(p) for p in parts]
    except ValueError:
        return None   # header or text line


def rezero():
    # Take a new baseline. NOTHING may be on the sensor.
    ser.reset_input_buffer()
    ser.write(b"b\n")
    end = time.time() + 3
    got = False
    while time.time() < end:
        line = ser.readline().decode(errors="ignore")
        if line.startswith(("Baseline", "ERROR", "WARNING")):
            print(line.strip()); got = True
    if not got:
        print("No baseline message received - check the firmware supports the 'b' command.")


def record(weight_g, seconds=None, settle=None, run=1, plot=True):
    # Put the weight on FIRST, then call this. Adds the readings to CSV_FILE.
    seconds = SECONDS if seconds is None else seconds
    settle  = SETTLE  if settle  is None else settle
    ser.reset_input_buffer()
    time.sleep(settle)              # let the reading settle
    ser.reset_input_buffer()
    rows, t0 = [], time.time()
    while time.time() - t0 < seconds:
        r = _parse(ser.readline().decode(errors="ignore"))
        if r:
            rows.append(r)
    if not rows:
        print("No readings - check the port, CSV_MODE = true on the ESP32, and the wiring.")
        return None
    new_file = not os.path.exists(CSV_FILE)
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(CSV_FILE, "a", newline="") as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(COLS)
        for t, ticks, dticks, cap, dpf, tmo in rows:
            w.writerow([stamp, run, weight_g, t, ticks, dticks, cap, dpf, int(tmo)])
    d = np.array([r[4] for r in rows])
    timeouts = int(sum(r[5] for r in rows))
    print(f"{weight_g} g: {len(rows)} readings, change = {d.mean():+.4f} pF (spread ±{d.std():.4f})"
          + (f"  [{timeouts} timeouts]" if timeouts else ""))
    if plot:
        plt.figure(figsize=(8, 2.5))
        plt.plot(np.array([r[0] for r in rows]) - rows[0][0], d)
        plt.xlabel("Time (s)"); plt.ylabel("Change (pF)"); plt.title(f"{weight_g} g (run {run})")
        plt.grid(alpha=0.3); plt.show()
    return d.mean()


def calibrate(weights=None, repeats=None):
    # Guided session: prompts you to place each weight, then records it.
    weights = WEIGHTS_G if weights is None else weights
    repeats = REPEATS   if repeats is None else repeats
    print(f"Saving to {os.path.abspath(CSV_FILE)}")
    for run in range(1, repeats + 1):
        print(f"\n===== Run {run} of {repeats} =====")
        input("Remove everything from the sensor, then press Enter to re-zero...")
        rezero()
        for wt in weights:
            msg = (f"\nPlace {wt} g on the sensor (centered, hands off), then press Enter "
                   "(type 's' to skip, 'q' to quit): ")
            if wt == 0:
                msg = "\nLeave the sensor EMPTY and press Enter to record 0 g (s = skip, q = quit): "
            ans = input(msg).strip().lower()
            if ans == "q":
                print("Stopped."); return
            if ans == "s":
                continue
            record(wt, run=run, plot=False)
    print("\nDone! Run Step 7 to see the results.")

print("Helpers loaded.")

## Step 5 — Zero the sensor
**Nothing on the sensor.** This sets the reference point every weight is compared to. Re-run it any time the empty reading drifts (temperature, moving cables, etc.).

In [ ]:
rezero()

## Step 6 — Record your weights
Choose **one** of the two options.

### Option A (recommended): guided session
Run the cell below. A text box appears under it each time — place the weight it asks for, take your hands off, and press **Enter**. Type `s` to skip a weight or `q` to stop early.

**Good practice:**
- Place each weight in the **same spot** (centered) every time — position changes the reading.
- Don't touch the sensor, table, or cable while it's recording.
- Lower weights gently; don't drop them.

In [ ]:
calibrate()

### Option B: one weight at a time
Useful for re-doing a single weight or adding an extra one. Put the weight on, then run `record(weight_in_grams)`. Copy the cell for each weight. The plot shows the reading over time — it should be a flat line with small noise. A slope means it hadn't settled (increase `SETTLE`); spikes mean something was bumped.

In [ ]:
# Example: put 250 g on the sensor, then run
# record(250)

## Step 7 — Results and calibration curve
This reads everything in `CSV_FILE`, averages the readings at each weight, and fits two curves:
- **Linear** (`pF = a·g + b`) — simple, but capacitive sensors often flatten out at higher loads.
- **Quadratic** (`pF = a·g² + b·g + c`) — usually follows that curve better.

**R²** tells you how well each curve fits (1.0 = perfect). The error bars show the spread of the readings; if you did multiple runs, the dots from each run are shown too so you can spot drift or hysteresis.

The plot is saved as a PNG next to the CSV, and the fit is saved to a `.json` file for Step 8.

In [ ]:
import json

data, by_run = {}, {}
with open(CSV_FILE) as f:
    for row in csv.DictReader(f):
        wt, v = float(row["weight_g"]), float(row["delta_pF"])
        run = int(row.get("run") or 1)
        data.setdefault(wt, []).append(v)
        by_run.setdefault((run, wt), []).append(v)

w = np.array(sorted(data))
m = np.array([np.mean(data[k]) for k in w])
s = np.array([np.std(data[k]) for k in w])

print(f"{'weight':>8}  {'mean change':>12}  {'spread':>8}  {'n':>6}")
for wi, mi, si in zip(w, m, s):
    print(f"{wi:7.0f} g  {mi:+11.4f} pF  ±{si:.4f}  {len(data[wi]):6d}")

def r2(y, yhat):
    ss_res = np.sum((y - yhat) ** 2); ss_tot = np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")

fits = {}
plt.figure(figsize=(7, 4.5))
runs = sorted({r for r, _ in by_run})
if len(runs) > 1:
    for r in runs:
        pts = sorted((wt, np.mean(v)) for (rr, wt), v in by_run.items() if rr == r)
        plt.plot(*zip(*pts), ".", alpha=0.5, label=f"run {r}")
plt.errorbar(w, m, yerr=s, fmt="o", color="k", capsize=3, label="mean ± spread")
xs = np.linspace(w.min(), w.max(), 200)
for deg, name, style in [(1, "linear", "--"), (2, "quadratic", "-")]:
    if len(w) > deg:
        c = np.polyfit(w, m, deg)
        fits[name] = {"coeffs": c.tolist(), "r2": float(r2(m, np.polyval(c, w)))}
        plt.plot(xs, np.polyval(c, xs), style, label=f"{name} (R² = {fits[name]['r2']:.3f})")
        print(f"\n{name} fit: coefficients {np.round(c, 8).tolist()}  R² = {fits[name]['r2']:.4f}")
if "linear" in fits:
    print(f"Average sensitivity: {fits['linear']['coeffs'][0]*1000:.4f} pF per kg")

plt.xlabel("Weight (g)"); plt.ylabel("Change in capacitance (pF)")
plt.title(f"Calibration: {os.path.basename(CSV_FILE)}")
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout()
png = os.path.splitext(CSV_FILE)[0] + ".png"
plt.savefig(png, dpi=150); plt.show()

cal_file = os.path.splitext(CSV_FILE)[0] + "_fit.json"
with open(cal_file, "w") as f:
    json.dump({"source": CSV_FILE, "weights_g": w.tolist(), "mean_pF": m.tolist(),
               "spread_pF": s.tolist(), "fits": fits}, f, indent=2)
print(f"\nSaved plot -> {png}\nSaved fit  -> {cal_file}")

## Step 8 — Convert a new reading to grams
Use the fitted curve backwards: give it a capacitance change (pF) and it returns the estimated weight. It only trusts values inside the range you calibrated — outside it, it warns you, because the curve isn't valid there.

The last line takes a live 3-second reading and converts it, so you can put an unknown object on the sensor and check.

In [ ]:
def pF_to_grams(delta_pF, model="quadratic"):
    c = np.array(fits[model]["coeffs"])
    lo, hi = w.min(), w.max()
    grid = np.linspace(lo, hi, 5000)
    curve = np.polyval(c, grid)
    if not (curve.min() <= delta_pF <= curve.max()):
        print(f"Warning: {delta_pF:+.4f} pF is outside the calibrated range "
              f"({curve.min():+.4f} to {curve.max():+.4f} pF).")
    return float(grid[np.argmin(np.abs(curve - delta_pF))])

print(f"0.15 pF -> {pF_to_grams(0.15):.0f} g")

# Live check: put something on the sensor, then uncomment:
# live = record("unknown", seconds=3, plot=False)
# print(f"Estimated weight: {pF_to_grams(live):.0f} g")

## Step 9 — Disconnect
Run this when you're done so the port is free for the Arduino IDE or another notebook.

**Starting a new test?** Change `CSV_FILE` in Step 2, run Steps 2–4 again, and continue.

In [ ]:
ser.close()
print("Port closed.")

In [ ]:
# Optional: open the CSV file (Windows only)
print(os.path.abspath(CSV_FILE))
if os.name == "nt":
    os.startfile(os.path.abspath(CSV_FILE))